**Import packages and dependecies**

In [ ]:
%pip install -q -e ..

In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tda.rips import VietorisRips
from tda.plotting import plot_barcode, plot_persistence_diagram

### 1. Load in data inputs

The Community-Level Health (CLH) database is maintained by the Agency for Healthcare Research and Quality (ARHQ). It provides data on healthcare outcomes and social determinants of health for geographic areas at a county, zip code, and census tract level. 

The fields in the CLH database can be categorized into 5 topic areas: 
1. Demographics (e.g. age, veteran status)
2. Economics (e.g. income, unemployment rate)
3. Physical infrastructure (e.g.internet connectivity, food and excercise access)
4. Education
5. Health (e.g. health providers, health conditions)

Users can download the raw data files here: https://www.ahrq.gov/data/innovations/clh-data.html

In [3]:
# Root location of project folders
root_path = str(Path.cwd().parent)

# Location of the CLH codebook file
codebook_path = fr"{root_path}\data\community_level_health_database\clh_2023_codebook_2_0.xlsx"
# Location of the CLH Excel file
data_path = fr"{root_path}\data\community_level_health_database\clh_2023_zipcode_2_0.xlsx"

# Read in files
codebook = pd.read_excel(codebook_path, sheet_name="Zipcode")
data = pd.read_excel(data_path, sheet_name="Data")

In [6]:
# Show data
display(codebook)

,Domain,Topic,Variable Name,Variable Label,Data Source,Type of Data (Numeric or Character),SAS Length,SAS Character,Position on the File
0,1. Demographics,Households,ACS_AVG_HH_SIZE_ZC,Average household size (ZCTA level),ACS,num,8,12.2,29
1,2. Economics,Income,ACS_GINI_INDEX_ZC,Gini index of income inequality (ZCTA level),ACS,num,8,6.2,181
2,2. Economics,Income,ACS_MDN_GRNDPRNT_INC_ZC,Median income of grandparent householder and/o...,ACS,num,8,16.2,183
3,2. Economics,Income,ACS_MDN_GRNDPRNT_NO_PRNT_INC_ZC,Median income of grandparent householder and/o...,ACS,num,8,16.2,182
4,4. Physical infrastructure,Housing and mobility,ACS_MDN_OWNER_COST_MORTGAGE_ZC,Median selected monthly owner costs for houses...,ACS,num,8,16.2,242
...,...,...,...,...,...,...,...,...,...
350,6. Geography,NaN,STATEFIPS,State FIPS Code (2-digit),NaN,char,2,$2.,2
351,6. Geography,NaN,TERRITORY,"Territory indicator (1= U.S. Territory, 0= U.S...",NaN,num,3,1.,7
352,Identifier,NaN,YEAR,File year,NaN,num,4,4.,1
353,6. Geography,NaN,ZCTA,"ZIP Code Tabulation Area, 5-digits",NaN,char,5,$5.,4


In [7]:
# Show data
display(data.head(10))

,YEAR,STATEFIPS,ZIPCODE,ZCTA,STATE,REGION,TERRITORY,POINT_ZIP,ACS_TOT_POP_WT_ZC,ACS_TOT_POP_US_ABOVE1_ZC,...,PC_PCT_MCARE_MAY_ACPT_APPRVD_AMT,POS_DIST_ED_ZP,POS_DIST_MEDSURG_ICU_ZP,POS_DIST_TRAUMA_ZP,POS_DIST_PED_ICU_ZP,POS_DIST_OBSTETRICS_ZP,POS_DIST_CLINIC_ZP,POS_DIST_ALC_ZP,CEN_AIAN_NH_IND,MISSING_ACS_DATA
0,2023,1,35004,35004,Alabama,South,0,0,11558.0,11419.0,...,5.56,9.81,11.30,15.40,29.70,9.81,9.83,18.39,0,0
1,2023,1,35005,35005,Alabama,South,0,0,8143.0,8098.0,...,NaN,1.57,7.27,13.25,24.99,9.27,6.58,9.50,0,0
2,2023,1,35006,35006,Alabama,South,0,0,3599.0,3574.0,...,NaN,13.20,13.20,24.89,24.89,13.20,14.37,23.51,0,0
3,2023,1,35007,35007,Alabama,South,0,0,27935.0,27536.0,...,2.37,0.89,0.89,14.53,0.89,0.89,0.72,18.32,0,0
4,2023,1,35137,35007,Alabama,South,0,1,27935.0,27536.0,...,NaN,2.79,2.79,15.60,2.79,2.79,2.71,20.06,0,0
5,2023,1,35144,35007,Alabama,South,0,1,27935.0,27536.0,...,NaN,3.99,3.99,16.55,3.99,3.99,3.76,21.18,0,0
6,2023,1,35010,35010,Alabama,South,0,0,19619.0,19370.0,...,4.39,2.18,2.18,25.24,36.07,2.18,0.78,25.24,0,0
7,2023,1,35011,35010,Alabama,South,0,1,19619.0,19370.0,...,NaN,1.87,1.87,24.64,35.58,1.87,0.85,24.64,0,0
8,2023,1,35013,35013,Alabama,South,0,1,122.0,122.0,...,NaN,6.82,6.82,21.93,55.18,21.76,3.67,21.93,0,0
9,2023,1,35014,35014,Alabama,South,0,0,3353.0,3353.0,...,NaN,9.26,9.26,12.13,36.59,9.26,8.56,12.13,0,0


### 2. Preprocessing

**Remove columns with high missing rate**

In [35]:
# Initialize parameters
columns_list = data.columns
tot_n_rows = data.shape[0]
missing_threshold = 0.4 # Columns w/ missing rate less than or equal to threshold are kept

missing_count_df = pd.DataFrame()
for column in columns_list:

    # Count the number of rows with null for given column
    missing_count = data[column].isna().sum()
    df = pd.DataFrame({
        "Variable Name": [column],
        "n_missing": [missing_count],
        "p_missing": [missing_count/tot_n_rows]
    })

    # Append result
    missing_count_df = pd.concat([missing_count_df, df])


# Merge on domain information
missing_count_df = codebook[["Variable Name", "Domain"]].merge(missing_count_df, on="Variable Name", how="left")

# Count the number of columns available in each domain before and after removing those that did not meet the missing rate threshold
all_columns_count_by_domain = (
    missing_count_df
    .groupby("Domain")
    ["Variable Name"].count()
    .reset_index()
    .rename(columns={"Variable Name": "tot_n_columns"})
)
missing_columns_count_by_domain = (
    missing_count_df[missing_count_df["p_missing"] <= missing_threshold]
    .groupby("Domain")
    ["Variable Name"].count()
    .reset_index()
    .rename(columns={"Variable Name": "n_columns_meet_threshold"})
)

# Final summary table showing the number of columns remaining
fin_columns_count_by_domain = (
    all_columns_count_by_domain
    .merge(missing_columns_count_by_domain, on="Domain", how="left")
    .fillna(0.0)
)
fin_columns_count_by_domain["difference"] = fin_columns_count_by_domain["tot_n_columns"] - fin_columns_count_by_domain["n_columns_meet_threshold"]
display(fin_columns_count_by_domain)

# Remove columns
columns_failed_threshold_list = missing_count_df[missing_count_df["p_missing"] > missing_threshold]["Variable Name"].values
preprocessed_data = data[[c for c in columns_list if c not in columns_failed_threshold_list]]

,Domain,tot_n_columns,n_columns_meet_threshold,difference
0,1. Demographics,124,120,4
1,2. Economics,81,69,12
2,3. Education,10,10,0
3,4. Physical infrastructure,87,66,21
4,5. Health,41,31,10
5,6. Geography,10,10,0
6,Identifier,2,2,0
